In [2]:
# Install the Google GenAI integration
!pip install -qU langgraph langchain-google-genai langchain-community langchain-core tiktoken faiss-cpu

import os
from google.colab import userdata

# Request the API Keys from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Multi-Agent Communication Router_Track_A_Gemini"

print("Environment setup complete. LangSmith tracing is active.")

Environment setup complete. LangSmith tracing is active.


In [4]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load Documents
mock_data = [
    Document(page_content="Policy: All vacation requests must be submitted via email to HR 14 days in advance.", metadata={"source": "hr_policy"}),
    Document(page_content="Contact Info: HR email is hr@company.com. IT support is it@company.com.", metadata={"source": "directory"}),
    Document(page_content="Meeting Room A is located on the 2nd floor and has a projector.", metadata={"source": "facilities"})
]

# 2. Split Documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
splits = text_splitter.split_documents(mock_data)

# 3. Embed and Store
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vectorstore = FAISS.from_documents(splits, embeddings)

# 4. Create Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

print(f"RAG Pipeline Ready. Stored {len(splits)} document chunks.")

RAG Pipeline Ready. Stored 3 document chunks.


In [5]:
from langgraph.func import task
from langgraph.types import interrupt
from langchain_core.runnables.config import RunnableConfig
import random

# Error Handling Strategy 1: Transient Retry
# This task will automatically retry if a transient error occurs during the API call.
@task(retry_policy={"max_attempts": 3, "initial_interval": 1.0})
def retrieve_knowledge(query: str, config: RunnableConfig) -> str:
    """Retrieves context from the RAG pipeline."""
    docs = retriever.invoke(query)
    if not docs:
        raise ValueError("Transient error: Vector store timeout.") # Simulating a transient failure
    return docs[0].page_content

@task
def draft_email(topic: str, recipient: str) -> str:
    """Drafts an email based on a topic."""
    return f"Drafted Email to {recipient}: Regarding {topic}, please review."

# Error Handling Strategy 2: User-fixable interrupt
@task
def send_email_with_approval(draft: str) -> str:
    """Pauses execution to ask a human for approval before sending."""
    # The interrupt function pauses the graph and surfaces this payload to the user
    user_approval = interrupt({"action": "approve_email", "draft": draft})

    if user_approval.get("approved") == True:
        return "Email successfully sent!"
    else:
        return "Email sending cancelled by user."

@task
def schedule_calendar_event(title: str, time: str) -> str:
    """Schedules a calendar event."""
    return f"Calendar event '{title}' scheduled for {time}."

In [6]:
from langgraph.func import entrypoint
from langgraph.checkpoint.memory import MemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize our LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)
checkpointer = MemorySaver()

# Define the Orchestrator-Worker pattern using the entrypoint decorator
@entrypoint(checkpointer=checkpointer)
def supervisor_agent(messages: list) -> str:
    """
    Supervisor Agent that routes requests to either the RAG/Email pipeline or Calendar pipeline.
    """
    user_input = messages[-1].content

    # Simple semantic routing for demonstration
    if "vacation" in user_input.lower() or "email" in user_input.lower():
        # Delegate to RAG and Email Worker
        policy_info = retrieve_knowledge(user_input).result()
        draft = draft_email(topic=policy_info, recipient="hr@company.com").result()

        # Require Human-in-the-loop approval before sending
        send_result = send_email_with_approval(draft).result()
        return send_result

    elif "schedule" in user_input.lower() or "meeting" in user_input.lower():
        # Delegate to Calendar Worker
        result = schedule_calendar_event(title="Team Sync", time="Tomorrow at 10 AM").result()
        return result

    else:
        return "I can only help with HR emails and scheduling meetings."

In [7]:
from langgraph.types import Command
from langchain_core.messages import HumanMessage

# We use a new thread_id so the agent doesn't remember the previous test
config = {"configurable": {"thread_id": "interactive_session_01"}}
initial_input = [HumanMessage(content="I want to request a vacation.")]

print("--- STARTING AGENT ---")
# First invocation: The agent will run until it hits the interrupt()
for chunk in supervisor_agent.stream(initial_input, config=config):
    print(chunk)

print("\n--- GRAPH PAUSED FOR HUMAN APPROVAL ---")

# The code will completely freeze here until you type a response and press Enter.
user_choice = input("The agent wants to send an email to HR. Type 'yes' to approve or 'no' to cancel: ")

# We translate your typed answer into the payload the agent expects
if user_choice.lower().strip() == 'yes':
    resume_payload = {"approved": True}
else:
    resume_payload = {"approved": False}

print("\n--- RESUMING AGENT ---")
# Second invocation: Resume the graph using the Command object
for chunk in supervisor_agent.stream(Command(resume=resume_payload), config=config):
    print(chunk)

print("\n--- AGENT FINISHED ---")

--- STARTING AGENT ---
{'retrieve_knowledge': 'Policy: All vacation requests must be submitted via email to HR 14 days in advance.'}
{'draft_email': 'Drafted Email to hr@company.com: Regarding Policy: All vacation requests must be submitted via email to HR 14 days in advance., please review.'}
{'__interrupt__': (Interrupt(value={'action': 'approve_email', 'draft': 'Drafted Email to hr@company.com: Regarding Policy: All vacation requests must be submitted via email to HR 14 days in advance., please review.'}, id='1c83ab83f0949073ba6c6e0eb3e7400f'),)}

--- GRAPH PAUSED FOR HUMAN APPROVAL ---
The agent wants to send an email to HR. Type 'yes' to approve or 'no' to cancel: yes

--- RESUMING AGENT ---
{'send_email_with_approval': 'Email successfully sent!'}
{'supervisor_agent': 'Email successfully sent!'}

--- AGENT FINISHED ---
